# Multi Agent Implementation for Overcooked


En este proyecto desarrollamos un juego cooperativo al estilo *Overcooked* en el que dos agentes deben colaborar para preparar sopas. Vamos a entrenarlos con el algoritmo **MAPPO** (*Multi-Agent Proximal Policy Optimization*) y compararemos visualmente su comportamiento antes y después del entrenamiento.

### ¿Qué es un sistema multiagente (MAS)?
Un MAS está compuesto por varios agentes autónomos que interactúan en un entorno común. Se clasifican por su estructura de recompensas:

| Tipo | Recompensas | Ejemplo |
|---|---|---|
| **Cooperativo** | Compartida entre todos los agentes | Overcooked, fútbol robótico |
| **Competitivo** | Suma cero | Ajedrez, Go |
| **Mixto** | Cooperación parcial dentro de equipos | StarCraft, League of Legends |

Overcooked es **puramente cooperativo**: ambos cocineros reciben la misma recompensa cuando el equipo entrega una sopa.

### ¿Por qué MAPPO y no PPO independiente?
Si entrenásemos a cada agente con PPO de forma independiente, el entorno parecería **no estacionario** desde la perspectiva de cada uno (porque el otro agente cambia su política durante el entrenamiento), rompiendo las garantías de PPO. MAPPO resuelve esto con el paradigma **CTDE** (*Centralized Training, Decentralized Execution*):

- **Entrenamiento centralizado**: el crítico ve información global para estimar mejor el valor.
- **Ejecución descentralizada**: en *inference* cada agente actúa solo con su observación local.

Además usamos **parameter sharing**: una única red de política compartida entre `agent_0` y `agent_1`. Es estándar en entornos cooperativos homogéneos porque (1) reduce el número de parámetros, (2) duplica la eficiencia muestral —cada experiencia entrena a ambos roles— y (3) induce coordinación implícita.

### Estructura del notebook
1. Instalación e imports.
2. Construcción del entorno como `ParallelEnv` de PettingZoo.
3. Adaptación a `MultiAgentEnv` de Ray RLlib.
4. Configuración de MAPPO con hiperparámetros explicados y modificables.
5. **Entrenamiento** del agente compartido + curva de aprendizaje.
6. **Evaluación cuantitativa**: aleatoria vs MAPPO.
7. **Renderizado a MP4**: comparación visual aleatoria vs MAPPO.

In [ ]:
!pip install overcooked-ai
!pip install pettingzoo
!pip install "ray[rllib]"
!pip install opencv-python imageio imageio-ffmpeg
!pip install pygame
!pip install gymnasium

## 1. Importación de librerías

Importamos las librerías necesarias para:

- **Overcooked-AI**: generación del entorno y la dinámica del juego (MDP).
- **PettingZoo (`ParallelEnv`)**: interpretación multiagente del entorno (acciones simultáneas de los dos cocineros).
- **Ray RLlib**: entrenamiento mediante MAPPO (Multi-Agent PPO con *parameter sharing*).
- **pygame + imageio**: captura de frames y exportación a MP4.

Forzamos el driver de vídeo `dummy` para que `pygame` funcione sin display (necesario en notebooks/headless).

In [ ]:
import os
# pygame en modo headless (para renderizar sin abrir ventana)
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

import numpy as np
import pygame
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces

# Overcooked-AI
from overcooked_ai_py.mdp.overcooked_mdp import OvercookedGridworld
from overcooked_ai_py.mdp.overcooked_env import OvercookedEnv
from overcooked_ai_py.visualization.state_visualizer import StateVisualizer

# PettingZoo
from pettingzoo import ParallelEnv

# Ray / RLlib
import ray
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.env.multi_agent_env import MultiAgentEnv
from ray.tune.registry import register_env

# Renderizado a MP4
import imageio

SEED = 42
np.random.seed(SEED)
print("Imports OK. Ray:", ray.__version__)

## 2. Creación del entorno Overcooked como entorno multiagente

Overcooked-AI expone su entorno a través de `OvercookedGridworld` (MDP) y `OvercookedEnv`. Para tratarlo como multiagente envolvemos esa API en una subclase de `pettingzoo.ParallelEnv` con **dos agentes** (`agent_0`, `agent_1`) que actúan **simultáneamente** en cada paso, lo cual es la formulación natural para entornos cooperativos como Overcooked.

- **Observación**: usamos el encoding `lossless_state_encoding_mdp` (tensor por agente).
- **Acciones**: 6 acciones discretas (4 movimientos, quieto, interactuar).
- **Recompensa**: cooperativa compartida + *reward shaping* por agente cuando está disponible.
- **Render**: imagen RGB obtenida del `StateVisualizer` (pygame).

In [ ]:
LAYOUT_NAME = "forced_coordination"   
HORIZON = 400                  # pasos máximos por episodio


class OvercookedParallelEnv(ParallelEnv):
    """Wrapper PettingZoo ParallelEnv para Overcooked-AI (2 cocineros cooperativos)."""

    metadata = {"render_modes": ["rgb_array"], "name": "overcooked_v0"}

    # 6 acciones discretas en Overcooked: N, S, E, O, Quieto, Interactuar
    NUM_ACTIONS = 6

    def __init__(self, layout_name=LAYOUT_NAME, horizon=HORIZON, render_mode="rgb_array"):
        self.layout_name = layout_name
        self.horizon = horizon
        self.render_mode = render_mode

        self.mdp = OvercookedGridworld.from_layout_name(layout_name)
        self.base_env = OvercookedEnv.from_mdp(self.mdp, horizon=horizon)
        self.visualizer = StateVisualizer()

        self.possible_agents = ["agent_0", "agent_1"]
        self.agents = self.possible_agents[:]

        # Forma de la observación con el encoding "lossless" estándar de Overcooked-AI
        sample_obs = self.mdp.lossless_state_encoding_mdp(self.base_env.state)
        obs_shape = sample_obs[0].shape

        self._observation_spaces = {
            a: spaces.Box(low=0.0, high=255.0, shape=obs_shape, dtype=np.float32)
            for a in self.possible_agents
        }
        self._action_spaces = {
            a: spaces.Discrete(self.NUM_ACTIONS) for a in self.possible_agents
        }

    def observation_space(self, agent):
        return self._observation_spaces[agent]

    def action_space(self, agent):
        return self._action_spaces[agent]

    def _get_obs(self):
        obs_both = self.mdp.lossless_state_encoding_mdp(self.base_env.state)
        return {
            "agent_0": np.asarray(obs_both[0], dtype=np.float32),
            "agent_1": np.asarray(obs_both[1], dtype=np.float32),
        }

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)
        self.base_env.reset()
        self.agents = self.possible_agents[:]
        return self._get_obs(), {a: {} for a in self.agents}

    def step(self, actions):
        joint_action = (int(actions["agent_0"]), int(actions["agent_1"]))
        next_state, reward, done, info = self.base_env.step(joint_action)

        # Recompensa cooperativa compartida (incluye shaping si está disponible)
        shaped = info.get("shaped_r_by_agent", [0.0, 0.0]) if isinstance(info, dict) else [0.0, 0.0]
        rewards = {
            "agent_0": float(reward) + float(shaped[0]),
            "agent_1": float(reward) + float(shaped[1]),
        }
        terminations = {a: bool(done) for a in self.agents}
        truncations = {a: False for a in self.agents}
        infos = {a: dict(info) if isinstance(info, dict) else {} for a in self.agents}
        observations = self._get_obs()

        if done:
            self.agents = []

        return observations, rewards, terminations, truncations, infos

    def render(self):
        surface = self.visualizer.render_state(
            state=self.base_env.state,
            grid=self.mdp.terrain_mtx,
            hud_data={"timestep": self.base_env.t},
        )
        arr = pygame.surfarray.array3d(surface)
        return np.transpose(arr, (1, 0, 2))  # (W, H, 3) -> (H, W, 3)

    def close(self):
        pass


# Sanity check rápido
_env = OvercookedParallelEnv()
_obs, _ = _env.reset(seed=SEED)
print("Layout:", _env.layout_name)
print("Agentes:", _env.agents)
print("Obs shape (agent_0):", _obs["agent_0"].shape)
print("Action space:", _env.action_space("agent_0"))
print("Frame de render shape:", _env.render().shape)

## 3. Adaptación a `MultiAgentEnv` de Ray RLlib

Ray RLlib espera la interfaz `MultiAgentEnv` (con el flag especial `"__all__"` en los diccionarios `terminated` / `truncated`). Envolvemos nuestro `OvercookedParallelEnv` y registramos el entorno con el nombre `overcooked_multiagent` para poderlo usar desde la configuración de entrenamiento.

In [ ]:
class OvercookedRLlibEnv(MultiAgentEnv):
    """Adapta el ParallelEnv de PettingZoo a la interfaz MultiAgentEnv de RLlib."""

    def __init__(self, env_config=None):
        super().__init__()
        env_config = env_config or {}
        self.par_env = OvercookedParallelEnv(
            layout_name=env_config.get("layout_name", LAYOUT_NAME),
            horizon=env_config.get("horizon", HORIZON),
        )
        self._agent_ids = set(self.par_env.possible_agents)
        # En RLlib se asume mismo espacio para todos los agentes con parameter sharing
        self.observation_space = self.par_env.observation_space("agent_0")
        self.action_space = self.par_env.action_space("agent_0")

    def reset(self, *, seed=None, options=None):
        obs, infos = self.par_env.reset(seed=seed, options=options)
        return obs, infos

    def step(self, action_dict):
        obs, rew, term, trunc, info = self.par_env.step(action_dict)
        term["__all__"] = all(term.values()) if term else False
        trunc["__all__"] = all(trunc.values()) if trunc else False
        return obs, rew, term, trunc, info

    def render(self):
        return self.par_env.render()


def env_creator(env_config):
    return OvercookedRLlibEnv(env_config)


register_env("overcooked_multiagent", env_creator)
print("Entorno 'overcooked_multiagent' registrado en Ray.")

## 4. Configuración de MAPPO

MAPPO se implementa en RLlib como `PPOConfig` multiagente. Para un entorno cooperativo homogéneo como Overcooked usamos **parameter sharing**: ambos cocineros comparten la misma red de política, lo que reduce muestras necesarias y estabiliza el entrenamiento.

Agrupamos **todos los hiperparámetros** en un único diccionario `HYPERPARAMS` para que sea trivial experimentar (cambiar `lr`, `clip_param`, `entropy_coeff`, etc., y volver a entrenar). Cada parámetro va comentado con su efecto.

| Categoría | Parámetros | Efecto principal |
|---|---|---|
| Rollouts | `train_batch_size`, `num_env_runners`, `rollout_fragment_length` | Cuántos datos se recogen por iteración |
| Optimización | `lr`, `num_epochs`, `minibatch_size`, `grad_clip` | Velocidad y estabilidad del SGD |
| PPO | `clip_param`, `gamma`, `lambda_`, `vf_loss_coeff`, `entropy_coeff` | Forma del objetivo (sesgo/varianza, exploración) |

In [ ]:
# =====================================================================
# Hiperparámetros de MAPPO.  Modifica aquí para experimentar.
# =====================================================================
HYPERPARAMS = {
    # ---------- Entorno ----------
    "layout_name": LAYOUT_NAME,         # layout de Overcooked (definido arriba)
    "horizon": HORIZON,                 # pasos por episodio

    # ---------- Rollouts (recolección de experiencia) ----------
    "num_env_runners": 2,               # workers paralelos -> más muestras/seg
    "rollout_fragment_length": 200,     # pasos por worker antes de un batch
    "train_batch_size": 4000,           # muestras totales por iteración de train()

    # ---------- Optimización (SGD) ----------
    "lr": 3e-4,                         # learning rate de Adam
    "num_epochs": 10,                   # épocas de SGD por iteración
    "minibatch_size": 256,              # tamaño de minibatch dentro de cada época
    "grad_clip": 0.5,                   # clip de gradiente (estabilidad)

    # ---------- Objetivo PPO ----------
    "gamma": 0.99,                      # factor de descuento (horizonte efectivo ~100 pasos)
    "lambda_": 0.95,                    # GAE-lambda (compromiso sesgo/varianza de la ventaja)
    "clip_param": 0.2,                  # clip ratio de PPO (cuánto puede cambiar la política)
    "vf_loss_coeff": 0.5,               # peso del loss del crítico (valor)
    "entropy_coeff": 0.01,              # bonificación de entropía -> fomenta exploración

    # ---------- Recursos ----------
    "num_gpus": 0,                      # pon 1 si tienes GPU disponible
    "framework": "torch",

    # ---------- Entrenamiento ----------
    "num_iterations": 20,               # nº de llamadas a algo.train()
}


def build_mappo_config(hp=HYPERPARAMS):
    """Construye la configuración MAPPO con parameter sharing a partir de un dict de hiperparámetros."""
    temp_env = OvercookedRLlibEnv({"layout_name": hp["layout_name"], "horizon": hp["horizon"]})
    obs_space = temp_env.observation_space
    act_space = temp_env.action_space

    # Una sola política compartida para los dos agentes (parameter sharing)
    policies = {"shared_policy": (None, obs_space, act_space, {})}

    def policy_mapping_fn(agent_id, *args, **kwargs):
        return "shared_policy"

    config = (
        PPOConfig()
        .environment(
            env="overcooked_multiagent",
            env_config={"layout_name": hp["layout_name"], "horizon": hp["horizon"]},
            disable_env_checking=True,
        )
        .framework(hp["framework"])
        .multi_agent(
            policies=policies,
            policy_mapping_fn=policy_mapping_fn,
            policies_to_train=["shared_policy"],
        )
        .training(
            train_batch_size=hp["train_batch_size"],
            minibatch_size=hp["minibatch_size"],
            num_epochs=hp["num_epochs"],
            lr=hp["lr"],
            gamma=hp["gamma"],
            lambda_=hp["lambda_"],
            clip_param=hp["clip_param"],
            entropy_coeff=hp["entropy_coeff"],
            vf_loss_coeff=hp["vf_loss_coeff"],
            grad_clip=hp["grad_clip"],
        )
        .env_runners(
            num_env_runners=hp["num_env_runners"],
            rollout_fragment_length=hp["rollout_fragment_length"],
        )
        .resources(num_gpus=hp["num_gpus"])
    )
    return config


mappo_config = build_mappo_config()
print("Configuración MAPPO lista.")
print(f"  Layout            : {HYPERPARAMS['layout_name']}")
print(f"  Iteraciones       : {HYPERPARAMS['num_iterations']}")
print(f"  Workers paralelos : {HYPERPARAMS['num_env_runners']}")
print(f"  Batch / minibatch : {HYPERPARAMS['train_batch_size']} / {HYPERPARAMS['minibatch_size']}")
print(f"  lr / clip / ent   : {HYPERPARAMS['lr']} / {HYPERPARAMS['clip_param']} / {HYPERPARAMS['entropy_coeff']}")
print(f"  Política compartida entre agent_0 y agent_1 (parameter sharing)")

## 5. Entrenamiento con MAPPO

Lanzamos el entrenamiento llamando iterativamente a `algo.train()`. Cada iteración:

1. Recolecta `train_batch_size` muestras de los `num_env_runners` workers en paralelo.
2. Calcula las **ventajas** con GAE (`gamma`, `lambda_`).
3. Actualiza la **política compartida** con `num_epochs` épocas de PPO sobre minibatches, recortando el ratio según `clip_param` para evitar pasos demasiado grandes.

Guardamos la recompensa media por episodio (`episode_return_mean`) para graficar la curva de aprendizaje al final. En `forced_coordination` —un layout que **obliga** a los agentes a cooperar pasándose objetos por el mostrador— la baseline aleatoria suele estar cerca de 0 y MAPPO debería superarla claramente tras unas iteraciones.



In [ ]:
import shutil


def _extract_metric(result, *keys, default=float("nan")):
    """Busca una métrica en estructura anidada de Ray (env_runners.*, sampler_results.*, raíz)."""
    pools = [result, result.get("env_runners", {}), result.get("sampler_results", {})]
    for pool in pools:
        for k in keys:
            if k in pool and pool[k] is not None:
                return pool[k]
    return default


# Reinicia Ray si ya estaba arrancado para evitar conflictos al re-ejecutar la celda
if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True, log_to_driver=False)

algo = mappo_config.build()

learning_curve = []
print(f"Entrenando MAPPO durante {HYPERPARAMS['num_iterations']} iteraciones...\n")
print(f"{'iter':>4} | {'reward_mean':>11} | {'reward_max':>10} | {'ep_len':>6}")
print("-" * 45)
for i in range(HYPERPARAMS["num_iterations"]):
    result = algo.train()
    ep_rew_mean = _extract_metric(result, "episode_return_mean", "episode_reward_mean")
    ep_rew_max  = _extract_metric(result, "episode_return_max",  "episode_reward_max")
    ep_len_mean = _extract_metric(result, "episode_len_mean")
    learning_curve.append({"iter": i + 1, "reward": ep_rew_mean,
                           "reward_max": ep_rew_max, "length": ep_len_mean})
    print(f"{i+1:>4} | {ep_rew_mean:>11.3f} | {ep_rew_max:>10.3f} | {ep_len_mean:>6.1f}")

# Guarda checkpoint final
checkpoint_dir = os.path.abspath("./mappo_checkpoint")
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
ckpt = algo.save(checkpoint_dir)
print(f"\nCheckpoint guardado en: {checkpoint_dir}")

# Curva de aprendizaje
iters   = [x["iter"] for x in learning_curve]
rewards = [x["reward"] for x in learning_curve]
plt.figure(figsize=(8, 4))
plt.plot(iters, rewards, marker="o", linewidth=2)
plt.title("Curva de aprendizaje MAPPO en Overcooked")
plt.xlabel("Iteración")
plt.ylabel("Recompensa media por episodio")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Política entrenada y evaluación cuantitativa

Construimos una función `policy_fn(obs_dict) -> action_dict` que envuelve al algoritmo entrenado. Esta función representa la fase de **ejecución descentralizada**: aunque ambos agentes compartan los mismos pesos, cada uno decide únicamente con su propia observación local.

Después medimos `evaluate_policy` sobre varios episodios para comparar de forma objetiva contra la baseline aleatoria.

In [ ]:
def make_trained_policy_fn(algo, policy_id="shared_policy", deterministic=True):
    """Crea una policy_fn que usa el algoritmo entrenado para decidir acciones.

    deterministic=True -> usa la acción más probable (modo evaluación).
    deterministic=False -> muestrea de la distribución (más variedad, peor rendimiento).
    """
    def policy_fn(obs_dict):
        actions = {}
        for agent_id, obs in obs_dict.items():
            action = algo.compute_single_action(
                observation=obs,
                policy_id=policy_id,
                explore=not deterministic,
            )
            actions[agent_id] = int(action)
        return actions
    return policy_fn


def evaluate_policy(policy_fn, n_episodes=5, layout_name=HYPERPARAMS["layout_name"]):
    """Devuelve (media, desviación) del retorno por episodio sobre n_episodes."""
    env = OvercookedParallelEnv(layout_name=layout_name)
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=SEED + ep)
        done = False
        total = 0.0
        while not done:
            if policy_fn is None:
                actions = {a: env.action_space(a).sample() for a in env.agents}
            else:
                actions = policy_fn(obs)
            obs, rewards, terms, truncs, _ = env.step(actions)
            total += sum(rewards.values()) / 2.0
            done = all(terms.values()) or all(truncs.values())
        returns.append(total)
    return float(np.mean(returns)), float(np.std(returns))


# Construimos la policy_fn y comparamos cuantitativamente vs baseline aleatoria
trained_policy_fn = make_trained_policy_fn(algo)

mean_rand,  std_rand  = evaluate_policy(None,              n_episodes=5)
mean_mappo, std_mappo = evaluate_policy(trained_policy_fn, n_episodes=5)

print("Recompensa media por episodio (5 episodios):")
print(f"  Aleatoria : {mean_rand:>7.2f} ± {std_rand:.2f}")
print(f"  MAPPO     : {mean_mappo:>7.2f} ± {std_mappo:.2f}")
print(f"  Mejora    : x{(mean_mappo / max(mean_rand, 1e-6)):.2f}")

## 7. Renderizado a MP4 — comparación visual

Para visualizar el efecto del entrenamiento generamos **dos vídeos** sobre el mismo layout y la misma semilla:

1. `overcooked_random.mp4` — agentes con política aleatoria (baseline).
2. `overcooked_mappo.mp4` — agentes con la política entrenada por MAPPO.

La función `rollout_to_mp4` es reutilizable: acepta cualquier `policy_fn(obs) -> actions`, captura los frames RGB del `StateVisualizer` de Overcooked y los escribe en MP4 con `imageio + ffmpeg`. En el vídeo de MAPPO deberías ver a los dos cocineros coordinándose para recoger ingredientes, dejarlos en la olla, esperar la cocción y servir los platos.

In [ ]:
def rollout_to_mp4(policy_fn=None, output_path="overcooked_rollout.mp4",
                   fps=8, max_steps=None, layout_name=HYPERPARAMS["layout_name"], seed=SEED):
    """Ejecuta un episodio completo y guarda un MP4 con el renderizado del entorno.

    policy_fn: callable(obs_dict) -> action_dict.  None = política aleatoria.
    """
    env = OvercookedParallelEnv(layout_name=layout_name)
    obs, _ = env.reset(seed=seed)
    frames = [env.render()]
    steps = max_steps or env.horizon
    done = False
    t = 0
    total_reward = 0.0
    while not done and t < steps:
        if policy_fn is None:
            actions = {a: env.action_space(a).sample() for a in env.agents}
        else:
            actions = policy_fn(obs)
        obs, rewards, terminations, truncations, _ = env.step(actions)
        total_reward += sum(rewards.values()) / 2.0
        frames.append(env.render())
        done = all(terminations.values()) or all(truncations.values())
        t += 1

    writer = imageio.get_writer(output_path, fps=fps, codec="libx264",
                                quality=8, macro_block_size=1)
    for frame in frames:
        writer.append_data(frame.astype(np.uint8))
    writer.close()
    print(f"  {os.path.basename(output_path):<25} frames={len(frames):>3}  "
          f"pasos={t:>3}  reward={total_reward:.2f}")
    return output_path


# Generamos los dos vídeos para comparar visualmente
print("Generando vídeos de comparación...")
rollout_to_mp4(policy_fn=None,              output_path="overcooked_random.mp4")
rollout_to_mp4(policy_fn=trained_policy_fn, output_path="overcooked_mappo.mp4")
print("\nListo. Abre los dos MP4 lado a lado para comparar el comportamiento.")